# Day 2 — Reference Genome Alignment Benchmarking
### BWA-MEM · Bowtie2 · STAR (1-pass + 2-pass) · Minimap2

> **#30DaysOfBioinformatics** | SubhadipJana1409  
> Reference: Vasimuddin et al. (2019) *BWA-MEM2*, IEEE IPDPS · Dobin et al. (2013) *STAR*, Bioinformatics

**Spec Steps:**
1. Build indexes: BWA, Bowtie2, STAR with **GENCODE v44 GTF** (8,468 splice junctions)
2. Align 3 WGS samples (BWA-MEM, Bowtie2, Minimap2) + 3 RNA-seq samples (STAR 1p/2p, Bowtie2)
3. Compute: mapping rate, properly paired %, **chimeric rate**, **multi-mappers**
4. STAR 2-pass vs 1-pass: novel junction discovery & reclassification
5. Benchmark **wall-clock time + peak RSS memory**; performance matrix


## 1. Setup & Tool Versions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import subprocess, os
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

PROJECT_DIR = Path().resolve().parent
RES_DIR     = PROJECT_DIR / 'results'
BENCH_DIR   = RES_DIR / 'benchmarks'
JUNC_DIR    = RES_DIR / 'junctions'
FLAG_DIR    = RES_DIR / 'flagstats'

# Tool versions
tools = [
    ('bwa 2>&1 | grep Version', 'BWA-MEM (BWA-MEM2 proxy)'),
    ('bowtie2 --version | head -1', 'Bowtie2'),
    ('STAR --version', 'STAR'),
    ('minimap2 --version', 'Minimap2'),
    ('samtools --version | head -1', 'Samtools'),
]
for cmd, label in tools:
    v = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (v.stdout or v.stderr).strip()[:80]
    print(f'{label:<30}: {out}')

# STAR index — junctions loaded
star_log = RES_DIR / 'indexes' / 'star' / 'Log.out'
if star_log.exists():
    import re
    m = re.search(r'(\d+) collapsed junctions', star_log.read_text())
    print(f'\nSTAR index: {m.group(1) if m else "?"} splice junctions from GENCODE v44 GTF')


## 2. Load Benchmark Data (Steps 3 + 5)

In [ ]:
df = pd.read_csv(BENCH_DIR / 'benchmark_full.tsv', sep='\t')
summary = pd.read_csv(BENCH_DIR / 'benchmark_summary.tsv', sep='\t')

print(f'Samples: {len(df)} alignments across {df["aligner"].nunique()} aligners')
print(f'Aligners: {sorted(df["aligner"].unique())}')
print()
display(df[['aligner','sample','type','map_pct','pp_pct','chimeric','multi_mappers','wall_s','peak_rss_mb']])


## 3. Step 3 — Mapping Rate by Aligner

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dtype, title in [
    (axes[0], 'WGS',     'WGS — Mapping Rate'),
    (axes[1], 'RNA-seq', 'RNA-seq — Mapping Rate'),
]:
    sub = df[df['type'] == dtype]
    pivot = sub.pivot_table(index='sample', columns='aligner', values='map_pct')
    pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='white', colormap='tab10')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('Mapped reads (%)')
    ax.set_ylim(97, 101); ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Aligner', fontsize=8)
    ax.axhline(100, color='grey', linestyle='--', alpha=0.4, linewidth=0.8)

plt.suptitle('Step 3 — Mapping Rates (GENCODE v44 GTF STAR index)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'mapping_rates.png', bbox_inches='tight')
plt.show()


## 4. Step 3 — Properly Paired %

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dtype, title in [
    (axes[0], 'WGS',     'WGS — Properly Paired %'),
    (axes[1], 'RNA-seq', 'RNA-seq — Properly Paired %'),
]:
    sub = df[df['type'] == dtype]
    pivot = sub.pivot_table(index='sample', columns='aligner', values='pp_pct')
    pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='white', colormap='Set2')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('Properly paired (%)')
    ax.set_ylim(94, 101); ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Aligner', fontsize=8)
    ax.axhline(100, color='grey', linestyle='--', alpha=0.4)

plt.suptitle('Step 3 — Properly Paired Read Pairs', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'properly_paired.png', bbox_inches='tight')
plt.show()

print('Note: Bowtie2 WGS PP% ~97-98% vs BWA/Minimap2 100% reflects stricter')
print('      insert-size model in BWA/Minimap2 for simulated PE reads.')


## 5. Step 3 — Multi-mappers (NH:i > 1)

In [ ]:
rna_df = df[df['type'] == 'RNA-seq'].copy()

fig, ax = plt.subplots(figsize=(10, 4))
pivot = rna_df.pivot_table(index='sample', columns='aligner', values='multi_mappers', aggfunc='sum')
pivot.plot(kind='bar', ax=ax, width=0.65, edgecolor='white', colormap='Set1')
ax.set_title('RNA-seq Multi-mappers (NH:i > 1) — STAR vs Bowtie2', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Multi-mapped reads'); ax.tick_params(axis='x', rotation=0)
ax.legend(title='Aligner', fontsize=9)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'multimappers.png', bbox_inches='tight')
plt.savefig(BENCH_DIR / 'multimappers.png', bbox_inches='tight')
plt.show()

print('STAR reports multi-mappers because it allows each read to report all valid')
print('alignment positions (NH:i tag). Bowtie2 in default mode reports best hit only.')
print()
print(rna_df[['aligner','sample','multi_mappers','mm_pct']].to_string(index=False))


## 6. Step 5 — Performance Matrix (Wall-Clock + Peak Memory)

In [ ]:
palette = {'bwa_mem':'#4C72B0','bowtie2':'#DD8452','minimap2':'#55A868',
           'bowtie2_rna':'#E377C2','star_1pass':'#C44E52','star_2pass':'#8172B2'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, (dtype, sfx) in enumerate([('WGS','DNA-seq/WGS'), ('RNA-seq','RNA-seq')]):
    sub = df[df['type']==dtype].groupby('aligner')[['wall_s','peak_rss_mb']].mean().reset_index()
    colors = [palette.get(a,'#888') for a in sub['aligner']]

    # Wall-clock
    ax = axes[0][col]
    bars = ax.bar(sub['aligner'], sub['wall_s'], color=colors, edgecolor='white', width=0.6)
    for b,v in zip(bars, sub['wall_s']):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                f'{v:.1f}s', ha='center', fontweight='bold', fontsize=9)
    ax.set_title(f'Wall-Clock — {sfx}', fontweight='bold')
    ax.set_ylabel('Seconds (avg 3 samples)'); ax.tick_params(axis='x', rotation=15)

    # Memory
    ax = axes[1][col]
    rss_gb = sub['peak_rss_mb'] / 1024
    bars = ax.bar(sub['aligner'], rss_gb, color=colors, edgecolor='white', width=0.6)
    for b,v in zip(bars, rss_gb):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                f'{v:.2f}GB', ha='center', fontweight='bold', fontsize=9)
    ax.set_title(f'Peak RSS Memory — {sfx}', fontweight='bold')
    ax.set_ylabel('GB (avg 3 samples)'); ax.tick_params(axis='x', rotation=15)

plt.suptitle('Step 5 — Performance Matrix\n(GENCODE v44 GTF; chr22 synthetic; /usr/bin/time -v)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'performance_matrix.png', bbox_inches='tight')
plt.show()


## 7. Summary Benchmark Table

In [ ]:
display(summary)

print('\nKey observations:')
print('  WGS speed:   Minimap2 (17.7s) > BWA-MEM (31.8s) > Bowtie2 (46.8s)')
print('  WGS memory:  Bowtie2 (0.13GB) < BWA-MEM (0.27GB) < Minimap2 (0.38GB)')
print('  RNA-seq:     STAR 2-pass uses 1.62GB vs 1-pass 0.92GB (second genome in RAM)')
print('  BWA-MEM2 expected: ~11-16s at same 0.27GB (2-3x speedup, identical accuracy)')


## 8. Step 4 — STAR 1-pass vs 2-pass Junction Analysis

STAR `--twopassMode Basic` mechanics:
1. **Pass 1:** Align reads → discover splice junctions (`SJ.out.tab`)
2. **Genome re-indexing:** Rebuild STAR index with discovered junctions + GENCODE v44
3. **Pass 2:** Re-align reads against augmented index → previously novel junctions become annotated

Column `annotated` in `SJ.out.tab`: `0` = novel (not in GTF), `1` = annotated (in GTF or Pass-1 list)


In [ ]:
junc_tsv = JUNC_DIR / 'star_junction_comparison.tsv'
if junc_tsv.exists():
    jdf = pd.read_csv(junc_tsv, sep='\t')
    display(jdf)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    samples = jdf['sample'].tolist()
    x = range(len(samples)); w = 0.35

    ax = axes[0]
    ax.bar([i-w/2 for i in x], jdf['pass1_total'], w, label='1-pass', color='#C44E52', edgecolor='white')
    ax.bar([i+w/2 for i in x], jdf['pass2_total'], w, label='2-pass', color='#8172B2', edgecolor='white')
    ax.set_xticks(list(x)); ax.set_xticklabels(samples)
    ax.set_title('Total Junctions Detected', fontweight='bold')
    ax.set_ylabel('Count'); ax.legend(); ax.set_ylim(0, 5)

    ax = axes[1]
    n1 = jdf['pass1_novel'].tolist(); a1 = jdf['pass1_annotated'].tolist()
    n2 = jdf['pass2_novel'].tolist(); a2 = jdf['pass2_annotated'].tolist()
    ax.bar([i-w/2 for i in x], n1, w, label='Novel (1p)',     color='#C44E52', edgecolor='white')
    ax.bar([i-w/2 for i in x], a1, w, label='Annotated (1p)', color='#e8a09a', edgecolor='white', bottom=n1)
    ax.bar([i+w/2 for i in x], n2, w, label='Novel (2p)',     color='#8172B2', edgecolor='white')
    ax.bar([i+w/2 for i in x], a2, w, label='Annotated (2p)', color='#c4b5e0', edgecolor='white', bottom=n2)
    ax.set_xticks(list(x)); ax.set_xticklabels(samples)
    ax.set_title('Novel vs Annotated (GENCODE v44 ref)', fontweight='bold')
    ax.set_ylabel('Count'); ax.legend(fontsize=8); ax.set_ylim(0, 5)

    plt.suptitle('STAR 1-pass vs 2-pass — Junction Analysis\n(reclassification = 1p-novel → 2p-annotated)',
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'star_junction_analysis.png', bbox_inches='tight')
    plt.show()
else:
    print(f'Run pipeline first: {junc_tsv} not found')


## 9. Step 3 — Flagstat Deep-Dive

In [ ]:
def parse_flagstat(path):
    stats = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            val = int(parts[0])
            if 'in total'          in line: stats['total']           = val
            elif 'mapped ('        in line and 'primary' not in line: stats['mapped'] = val
            elif 'properly paired' in line: stats['properly_paired'] = val
            elif 'singletons'      in line: stats['singletons']      = val
            elif 'secondary'       in line: stats['secondary']       = val
            elif 'supplementary'   in line: stats['supplementary']   = val
    return stats

rows = []
for f in sorted(FLAG_DIR.glob('*.txt')):
    s = parse_flagstat(f)
    s['file'] = f.stem
    rows.append(s)

fdf = pd.DataFrame(rows).fillna(0).astype({'total':int,'mapped':int})
fdf = fdf[['file','total','mapped','properly_paired','secondary','supplementary','singletons']]
display(fdf)


## 10. Mapping Rate Heatmap — All Aligners × All Samples

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
pivot = df.pivot_table(index='aligner', columns='sample', values='map_pct')
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=97, vmax=101, linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Mapped reads (%)'},
            ax=ax, annot_kws={'size':9})
ax.set_title('Mapping Rate Heatmap — All Aligners × All Samples', fontweight='bold', pad=12)
ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'heatmap_mapping.png', bbox_inches='tight')
plt.show()


## 12. Insert Size Distribution

Insert size (template length, TLEN field) computed from properly-paired reads in each BAM.
Visualises whether each aligner accurately models the simulated insert size distribution.


In [ ]:
import matplotlib.gridspec as gridspec

# Load insert size data
idf = pd.read_csv(BENCH_DIR / 'insert_sizes_raw.tsv', sep='\t')

sample_pal = {'NA12878_S1':'#4C72B0','NA12878_S2':'#DD8452','NA12878_S3':'#55A868'}
rna_pal    = {'PBMC_S1':'#C44E52','PBMC_S2':'#8172B2','PBMC_S3':'#4C9BE8'}
pal = {'BWA-MEM':'#4C72B0','Bowtie2':'#DD8452','Minimap2':'#55A868','STAR-1pass':'#C44E52','STAR-2pass':'#8172B2'}

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# Panel A: WGS KDE (BWA-MEM)
ax1 = fig.add_subplot(gs[0,0])
for s, c in sample_pal.items():
    d = idf[(idf['type']=='WGS') & (idf['aligner']=='BWA-MEM') & (idf['sample']==s)]['insert_size']
    d.plot.kde(ax=ax1, label=f'{s} (μ={d.mean():.0f})', color=c, linewidth=2)
    ax1.axvline(d.mean(), color=c, linestyle='--', alpha=0.5)
ax1.set(title='WGS Insert Size — BWA-MEM\n(3 samples, KDE)', xlabel='Insert size (bp)', ylabel='Density')
ax1.set_xlim(100,700); ax1.legend(fontsize=8)

# Panel B: WGS boxplot per aligner
ax2 = fig.add_subplot(gs[0,1])
order = ['BWA-MEM','Bowtie2','Minimap2']
data  = [idf[(idf['type']=='WGS')&(idf['aligner']==a)]['insert_size'].sample(5000,random_state=42) for a in order]
bp = ax2.boxplot(data, tick_labels=order, patch_artist=True, notch=False,
                  medianprops=dict(color='black',linewidth=2), flierprops=dict(marker='.',markersize=1,alpha=0.2))
for patch,a in zip(bp['boxes'],order): patch.set_facecolor(pal[a]); patch.set_alpha(0.75)
ax2.set(title='WGS Insert Size by Aligner\n(boxplot, 5k reads)', xlabel='Aligner', ylabel='Insert size (bp)')
ax2.set_ylim(100,700)

# Panel C: RNA-seq KDE (STAR-2pass)
ax3 = fig.add_subplot(gs[1,0])
for s, c in rna_pal.items():
    d = idf[(idf['type']=='RNA-seq') & (idf['aligner']=='STAR-2pass') & (idf['sample']==s)]['insert_size']
    d.plot.kde(ax=ax3, label=f'{s} (μ={d.mean():.0f})', color=c, linewidth=2)
    ax3.axvline(d.mean(), color=c, linestyle='--', alpha=0.5)
ax3.set(title='RNA-seq Insert Size — STAR 2-pass\n(3 samples, KDE)', xlabel='Insert size (bp)', ylabel='Density')
ax3.set_xlim(50,600); ax3.legend(fontsize=8)

# Panel D: Summary table
ax4 = fig.add_subplot(gs[1,1]); ax4.axis('off')
rows = []
for a in ['BWA-MEM','Bowtie2','Minimap2','STAR-1pass','STAR-2pass']:
    sub = idf[idf['aligner']==a]['insert_size']
    if len(sub)==0: continue
    rows.append([a, idf[idf['aligner']==a]['type'].iloc[0],
                 f'{sub.mean():.0f} ± {sub.std():.0f}',
                 f'{sub.median():.0f}',
                 f'{sub.quantile(0.05):.0f}–{sub.quantile(0.95):.0f}'])
tbl = ax4.table(cellText=rows, colLabels=['Aligner','Context','Mean ± SD','Median','5th–95th pct'],
                cellLoc='center', loc='center', bbox=[0,0.1,1,0.85])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r,c),cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white',fontweight='bold')
    elif r%2==0: cell.set_facecolor('#f8f9fa')
    cell.set_edgecolor('#dee2e6')
ax4.set_title('Insert Size Summary', fontweight='bold', pad=10, y=1.0)

plt.suptitle('Insert Size Distribution — All Aligners and Sample Types', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'insert_size_distribution.png', bbox_inches='tight')
plt.show()


## 11. Key Findings & Interpretation

### Step 3 — Alignment Statistics
| Metric | Observation | Why |
|--------|-------------|-----|
| Chimeric reads | 0 for all aligners | Synthetic chr22 — single chromosome, no inter-chrom events |
| Multi-mappers | 0 (BWA/Bowtie2/Minimap2), 824–1350 (STAR) | STAR outputs all valid alignments (NH tag); others report best-hit only |
| PP% Bowtie2 WGS | ~97–98% vs 100% for BWA/Minimap2 | Different insert-size estimation models |

### Step 4 — STAR Junction Reclassification
| Result | Meaning |
|--------|--------|
| 1-pass novel → 2-pass annotated | GENCODE v44 junctions loaded in Pass-2 genome — previously unrecognised junctions now matched |
| Low total junction count | `wgsim` simulates reads from flat sequence — no exon models, so reads don't span splice sites |
| Real PBMC data expectation | ~50k–100k junctions; 2-pass recovers 5–15% more novel junctions |

### Step 5 — Performance
| Aligner | Speed rank | Memory rank | Use case |
|---------|:----------:|:-----------:|----------|
| Minimap2 | 🥇 17.7s | 3rd 0.38GB | WGS, versatile |
| BWA-MEM | 2nd 31.8s | 2nd 0.27GB | WGS gold standard |
| BWA-MEM2 (est.) | 🥇 ~12s | 2nd 0.27GB | WGS with SIMD speedup |
| Bowtie2 | 3rd 46.8s | 🥇 0.13GB | Memory-constrained |
| STAR 1-pass | 20.5s | 3rd 0.92GB | RNA-seq |
| STAR 2-pass | 41.8s | 4th 1.62GB | RNA-seq + novel junctions |
